# Desafio 3: Optimización de despliegue PON

Integrantes:
* Camila Herrera
* Gustavo Venegas
* Javier Cáceres

## Secciones del Jupyter

1. Preparar herramientas (instalaciones e imports).
2. Extraer topología de OPenStreetMap (OSMnx).
3. Formular y resolver un ILP (PuLP) para ubicación de splitters y asignación de usuarios.
4. Validar presupuesto óptico y latencia.
5. Implementar funciones y ejemplos de ejecución.

## 1. Preparación: Instalación e imports

In [1]:
# Celda de setup: instalar paquetes (ejecutar solo si es necesario)
# Crear enviroment con: python3 -m venv wdm
# !pip install osmnx networkx geopandas shapely pulp matplotlib


# Imports necesarios
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pulp
from shapely.geometry import Point


# Parámetros globales (se pueden ajustar)
COST_FIBER_PER_M = 1.0 # coste por metro (unidad monetaria arbitraria)
COST_SPLITTER = 200.0 # coste por splitter
SPLIT_RATIO = 32 # ratio típico 1:32
RMAX = 10000.0 # alcance máximo admisible (m). Ajustar según presupuesto óptico
FIBER_ATTEN_DB_PER_KM = 0.35
CONNECTOR_LOSS_DB = 0.5 # pérdida por conector (estimada)
SPLITTER_LOSS_TABLE = {2:3.5, 4:7.0, 8:10.5, 16:13.5, 32:17.5} # valores típicos de referencia


# Mostrar versión de paquetes
print('osmnx', ox.__version__)
print('networkx', nx.__version__)

osmnx 2.0.6
networkx 3.5


## 2. Extracción topológica y generación de usuarios

In [2]:
# Definir área de interés (ejemplo: Valparaiso, Chile). Cambiar por otra ciudad/área si se desea.
place_name = 'Valparaíso, Chile'

# Descargar la red vial (driving network) y proyectar a CRS métrico
G = ox.graph_from_place(place_name, network_type='drive')
G = ox.project_graph(G)

# Obtener nodos como GeoDataFrame (índice = ID real del grafo)
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Asegurarnos de que el índice sea del mismo tipo que G.nodes
assert set(nodes_gdf.index) == set(G.nodes), "Los índices de nodes_gdf no coinciden con G.nodes"

# Generar usuarios sintéticos directamente a partir de los índices válidos
num_users_example = 50
user_node_ids = np.random.choice(nodes_gdf.index, size=num_users_example, replace=False)

# Crear GeoDataFrame de usuarios basado en esos nodos
users_gdf = nodes_gdf.loc[user_node_ids, ['x', 'y']].copy()
users_gdf['geometry'] = users_gdf.apply(lambda r: Point(r.x, r.y), axis=1)
users_gdf = gpd.GeoDataFrame(users_gdf, geometry='geometry', crs=nodes_gdf.crs)
users_gdf['node_id'] = users_gdf.index  # estos índices son los IDs válidos del grafo

# Seleccionar candidatos a splitter: intersecciones con grado >= 3
candidate_nodes = nodes_gdf[nodes_gdf['street_count'] >= 3].copy()
candidate_nodes['node_id'] = candidate_nodes.index

print('Nodos totales:', len(nodes_gdf))
print('Usuarios muestreados:', len(users_gdf))
print('Nodos candidatos a splitter:', len(candidate_nodes))

Nodos totales: 7677
Usuarios muestreados: 50
Nodos candidatos a splitter: 5513


## 3. Precomputar rutas y distancias entre candidatos y usuarios

In [3]:
candidate_ids = list(candidate_nodes.index)
user_ids = list(users_gdf.index)

paths_edges = {}
path_lengths = {}

for u in user_ids:
    for s in candidate_ids:
        try:
            path_nodes = nx.shortest_path(G, source=s, target=u, weight='length')
            path_edges = list(zip(path_nodes[:-1], path_nodes[1:]))
            length = sum(
                G.get_edge_data(e[0], e[1])[0].get('length', 0)
                if isinstance(G.get_edge_data(e[0], e[1]), dict)
                else G.get_edge_data(e[0], e[1]).get('length', 0)
                for e in path_edges
            )
            paths_edges[(u, s)] = path_edges
            path_lengths[(u, s)] = length
        except (nx.NetworkXNoPath, KeyError):
            continue

print("Precomputadas rutas para pares user-candidate:", len(path_lengths))

Precomputadas rutas para pares user-candidate: 275150


## 4. Formulación ILP con PuLP

In [4]:
# Crear problema PuLP
prob = pulp.LpProblem('PON_Splitter_Placement', pulp.LpMinimize)

# Variables:
# xe_e = 1 si se instala fibra en la arista e (representada por (u,v) tuple)
edge_keys = set()
for edges in paths_edges.values():
    for e in edges:
        edge_keys.add(e)
edge_keys = list(edge_keys)

xe = pulp.LpVariable.dicts('x_edge', (range(len(edge_keys))), lowBound=0, upBound=1, cat='Binary')

# ys_s = 1 si se instala splitter en candidato s
ys = pulp.LpVariable.dicts('y_splitter', (candidate_ids), lowBound=0, upBound=1, cat='Binary')

# zu_s = 1 si usuario u asignado a splitter s
zu = pulp.LpVariable.dicts('assign', [(u,s) for u in user_ids for s in candidate_ids], lowBound=0, upBound=1, cat='Binary')

# Objetivo: coste fibra + coste splitters
# coste fibra: sumar longitud de cada arista * COST_FIBER_PER_M * xe
edge_length_map = {i: sum(G[u][v][0].get('length',0) for _ in [0]) for i,(u,v) in enumerate(edge_keys)}

fiber_cost_expr = pulp.lpSum([COST_FIBER_PER_M * edge_length_map[i] * xe[i] for i in range(len(edge_keys))])
splitter_cost_expr = pulp.lpSum([COST_SPLITTER * ys[s] for s in candidate_ids])
prob += fiber_cost_expr + splitter_cost_expr

# Restricciones:
# 1) cada usuario debe estar asignado a exactamente un splitter
for u in user_ids:
    prob += pulp.lpSum([zu[(u,s)] for s in candidate_ids if (u,s) in path_lengths]) == 1

# 2) capacidad de splitter
for s in candidate_ids:
    prob += pulp.lpSum([zu[(u,s)] for u in user_ids if (u,s) in path_lengths]) <= SPLIT_RATIO * ys[s]

# 3) alcance físico: si asignado entonces path_length <= RMAX (implementar prohibición si excede)
for (u,s), length in path_lengths.items():
    if length > RMAX:
        prob += zu[(u,s)] == 0

# 4) conectividad: si zu[(u,s)] == 1 entonces todos los edges on path must be installed
# Implementamos: for each (u,s) and for each edge e in its path: xe_edge_index >= zu[(u,s)]
edge_index_lookup = {edge_keys[i]: i for i in range(len(edge_keys))}
for (u,s), edges in paths_edges.items():
    for e in edges:
        idx = edge_index_lookup[e]
        prob += xe[idx] >= zu[(u,s)]

# Resolver (solver por defecto CBC)
prob.solve(pulp.PULP_CBC_CMD(msg=1))
print('Status:', pulp.LpStatus[prob.status])

# Extraer solución: splitters instalados y fibra instalada
installed_splitters = [s for s in candidate_ids if pulp.value(ys[s])>0.5]
installed_edges = [edge_keys[i] for i in range(len(edge_keys)) if pulp.value(xe[i])>0.5]
assignments = [(u,s) for (u,s) in zu.keys() if pulp.value(zu[(u,s)])>0.5]

print('Splitters instalados:', len(installed_splitters))
print('Aristas de fibra instaladas:', len(installed_edges))
print('Asignaciones válidas (ejemplo):', assignments[:10])

KeyboardInterrupt: 

## 5. Funciones de verificación física: presupuesto óptico y latencia

In [ ]:
def compute_optical_loss_for_path(path_length_m, split_ratio, connectors=2, splice_loss_db=0.1):
    """Calcula pérdida total estimada (dB) usando parámetros simplificados.
    - path_length_m: longitud en metros desde OLT hasta ONU
    - split_ratio: ratio del splitter (e.g., 32)
    - connectors: número de conectores en la ruta (estimado)
    - splice_loss_db: pérdida por empalme (estimado)
    """
    fiber_km = path_length_m/1000.0
    fiber_loss = fiber_km * FIBER_ATTEN_DB_PER_KM
    splitter_loss = SPLITTER_LOSS_TABLE.get(split_ratio, None)
    if splitter_loss is None:
        # interpolar o usar fórmula ideal: 10*log10(N)
        splitter_loss = 10*np.log10(split_ratio)
    connector_total = connectors * CONNECTOR_LOSS_DB
    # suponemos 1 splice por km (ejemplo)
    splice_total = fiber_km * splice_loss_db
    total_loss_db = fiber_loss + splitter_loss + connector_total + splice_total
    return total_loss_db

# Ejemplo: verificar para la primera asignación
if assignments:
    u,s = assignments[0]
    length_m = path_lengths[(u,s)]
    loss_db = compute_optical_loss_for_path(length_m, SPLIT_RATIO)
    print(f'Usuario {u} asignado a splitter {s}: longitud {length_m:.1f} m, pérdida estimada {loss_db:.2f} dB')

# Verificación frente a un presupuesto (ejemplo: GPON class B+ 28 dB)
GPON_CLASS_BPLUS_BUDGET_DB = 28.0
print('Cumple presupuesto B+?', loss_db <= GPON_CLASS_BPLUS_BUDGET_DB)

# Latencia de propagación: velocidad de la luz en fibra ~ c/n, n≈1.4682 => ~204,000 km/s => 4.9 µs/km
def propagation_latency_us(path_length_m, refractive_index=1.4682):
    speed_m_s = 3e8 / refractive_index
    latency_s = path_length_m / speed_m_s
    return latency_s*1e6

if assignments:
    latency_us = propagation_latency_us(length_m)
    print(f'Latencia de propagación: {latency_us:.2f} µs')

## 6. Visualización del resultado (mapa)

In [ ]:
# Dibujar grafo y resaltar aristas instaladas y nodos de splitter
fig, ax = ox.plot_graph(G, show=False, close=False)
# Resaltar aristas instaladas
for (u,v) in installed_edges:
    xs = [G.nodes[u]['x'], G.nodes[v]['x']]
    ys = [G.nodes[u]['y'], G.nodes[v]['y']]
    ax.plot(xs, ys, linewidth=2, alpha=0.8)
# Resaltar splitters
splitter_points = [Point(G.nodes[s]['x'], G.nodes[s]['y']) for s in installed_splitters]
if splitter_points:
    splitter_gdf = gpd.GeoDataFrame(geometry=splitter_points, crs=nodes_gdf.crs)
    splitter_gdf.plot(ax=ax, markersize=30)

plt.show()